In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import datetime

import IPython
import IPython.display
import matplotlib as mpl
import tensorflow as tf

In [ ]:
df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
date_time = pd.to_datetime(df['date'])
df = df.set_index('date')
df.drop(columns= 'Unnamed: 0', inplace= True)
df.drop(columns= 'Unnamed: 0.1', inplace= True)

df.head()

In [ ]:
df.describe().transpose()

In [ ]:
# drop columns if they exist (no error if already absent)
df.drop(columns=['radiation', 'ndvi', 'dewpoint'], inplace=True, errors='ignore')

In [ ]:
df.head()

In [ ]:
plot_cols = ['et_loss', 'precipitation', 'sca', 'dd', 'temp', 'runoff']
plot_features = df[plot_cols]
plot_features.index = date_time
_ = plot_features.plot(subplots=True)

plot_features = df[plot_cols][:480]
plot_features.index = date_time[:480]
_ = plot_features.plot(subplots=True)

In [ ]:
# we convert 0-12 to radians circle for a seasonal mapping (circular time encoding)
# ensure index is treated as datetime (works even if the Index is not recognized as DatetimeIndex)
month = pd.DatetimeIndex(df.index).month

df['year_sin'] = np.sin(2*np.pi*month/12)
df['year_cos'] = np.cos(2*np.pi*month/12)

In [ ]:
plt.plot(np.array(df['year_sin'])[:25])
plt.plot(np.array(df['year_cos'])[:25])
plt.xlabel('Time [h]')
plt.title('Time of month signal')

In [ ]:
# monthly data: 12 samples per year
fft = tf.signal.rfft(df['sca'])
f_per_dataset = np.arange(0, len(fft))

n_samples_m = len(df['sca'])   # number of monthly samples
months_per_year = 12
years_per_dataset = n_samples_m / months_per_year

# frequency in cycles per year
f_per_year = f_per_dataset * months_per_year / n_samples_m

plt.step(f_per_year, np.abs(fft.numpy()))
plt.xscale('log')
plt.ylim(0, 400000)
plt.xlim([0.1, max(plt.xlim())])
plt.xticks([1, 12], labels=['1/Year', '1/Month'])
plt.xlabel('Frequency (log scale, cycles/year)')

In [ ]:
df.head()

In [ ]:
column_indices = {name: i for i, name in enumerate(df.columns)}

n = len(df)
train_df = df[0:int(n*0.7)]
val_df = df[int(n*0.7):int(n*0.9)]
test_df = df[int(n*0.9):]

num_features = df.shape[1]